In [1]:
### Used to create single numerical hitter ratings
import pandas as pd
import numpy as np
from scipy.stats import norm
import uuid

# Load the CSV data
df = pd.read_csv('All_Hitters_2020_2024_with_Scores.csv')

# Clean data: Ensure numeric columns are properly typed, handle missing values
numeric_cols = ['WAR', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'BB', 'SO',
                'BA', 'OBP', 'SLG', 'OPS', 'OPS+', 'rOBA', 'Rbat+', 'TB', 'AAV']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Define features for scoring
features = ['WAR', 'OPS+', 'rOBA', 'PA', 'BA', 'OBP', 'SLG', 'HR', 'RBI', 'SB']
weights = {'WAR': 0.30, 'OPS+': 0.10, 'rOBA': 0.10, 'PA': 0.10,
           'BA': 0.05, 'OBP': 0.05, 'SLG': 0.05, 'HR': 0.05, 'RBI': 0.05, 'SB': 0.05}

# Normalize features to 0-1 scale (except WAR, which is used directly)
normalized_df = df.copy()
for feature in features:
    if feature != 'WAR':
        if df[feature].max() > df[feature].min():
            normalized_df[feature] = (df[feature] - df[feature].min()) / (df[feature].max() - df[feature].min())
        else:
            normalized_df[feature] = 0

# Compute cost-effectiveness: WAR per million dollars of AAV
df['CostEffectiveness'] = df['WAR'] / (df['AAV'] / 1_000_000).replace(0, 1)
# Cap extreme values to prevent skewing
df['CostEffectiveness'] = df['CostEffectiveness'].clip(lower=0, upper=df['CostEffectiveness'].quantile(0.95))
normalized_df['CostEffectiveness'] = (df['CostEffectiveness'] - df['CostEffectiveness'].min()) / \
                                    (df['CostEffectiveness'].max() - df['CostEffectiveness'].min())
weights['CostEffectiveness'] = 0.10

# Define recency weights for years
year_weights = {2024: 0.40, 2023: 0.25, 2022: 0.20, 2021: 0.10, 2020: 0.05}

# Aggregate data by player, applying recency weights
player_scores = []
for player in df['Player'].unique():
    player_data = df[df['Player'] == player].copy()
    if len(player_data) == 1:
        # Single-year player: Use normalized stats directly
        player_normalized = normalized_df[df['Player'] == player]
        composite_score = sum(weights[feat] * player_normalized[feat].iloc[0] for feat in weights)
    else:
        # Multi-year player: Weight stats by year
        weighted_stats = {feat: 0 for feat in weights}
        total_weight = sum(year_weights.get(year, 0) for year in player_data['Year'])
        if total_weight == 0:
            total_weight = 1  # Avoid division by zero
        for _, row in player_data.iterrows():
            year = row['Year']
            weight = year_weights.get(year, 0) / total_weight
            for feat in weights:
                weighted_stats[feat] += weight * normalized_df.loc[row.name, feat]
        composite_score = sum(weights[feat] * weighted_stats[feat] for feat in weights)
    player_scores.append({'Player': player, 'CompositeScore': composite_score})

# Create DataFrame of composite scores
scores_df = pd.DataFrame(player_scores)

# Transform scores to normal distribution (1-99)
mean_score = scores_df['CompositeScore'].mean()
std_score = scores_df['CompositeScore'].std()
scores_df['Z'] = (scores_df['CompositeScore'] - mean_score) / std_score
scores_df['Score_1_100'] = norm.cdf(scores_df['Z']) * 98 + 1  # Scale to 1-99
scores_df['Score_1_100'] = scores_df['Score_1_100'].round().astype(int)
scores_df['Score_1_100'] = scores_df['Score_1_100'].clip(1, 99)  # Ensure within bounds

# Merge scores back to original DataFrame
df = df.drop(columns=['Score_1_100'])
df = df.merge(scores_df[['Player', 'Score_1_100']], on='Player', how='left')

# Export the updated CSV
df.to_csv('All_Hitters_2020_2024_with_Updated_Scores.csv', index=False)

# For artifact: Return the updated CSV content as a string
with open('All_Hitters_2020_2024_with_Updated_Scores.csv', 'r') as f:
    updated_csv_content = f.read()

print(updated_csv_content)

Rk,Player,Age,Team,Lg,WAR,G,PA,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,BA,OBP,SLG,OPS,OPS+,rOBA,Rbat+,TB,GIDP,HBP,SH,SF,IBB,Pos,Awards,AAV,Year,Unnamed: 0,Year_Weight,Weighted_WAR,Salary_per_G,CostEffectiveness,Score_1_100
1,Aaron Hicks,30.0,NYY,AL,0.7,54.0,211.0,169.0,28.0,38.0,10.0,2.0,6.0,21.0,4.0,1.0,41.0,38.0,0.225,0.379,0.414,0.793,122.0,0.356,124.0,70.0,4.0,1.0,0.0,0.0,1.0,*8/HD,,3888888.0,2020,,1,0.7,72016.44444444444,0.18000004114286652,44
2,Aaron Judge,28.0,NYY,AL,1.1,28.0,114.0,101.0,23.0,26.0,3.0,0.0,9.0,22.0,0.0,1.0,10.0,32.0,0.257,0.336,0.554,0.891,143.0,0.379,145.0,56.0,5.0,2.0,0.0,0.0,0.0,9/DH,,3148148.0,2020,,1,1.1,112433.85714285714,0.34941178114878974,99
3,Aaron Whitefield,23.0,MIN,AL,0.1,3.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,0.0,-125.0,0.0,0.0,0.0,0.0,0.0,0.0,/8H,,15000.0,2020,,1,0.1,5000.0,6.666666666666667,24
4,Abraham Almonte,31.0,SDP,NL,-0.2,7.0,13.0,11.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,4.0,0.091,0.231,0.091,0.322,-5.0,0.156,2.0,

In [14]:
### Used to create single numerical pitcher ratings
import pandas as pd
import numpy as np
from scipy.stats import norm
import uuid

# Load the CSV data
df = pd.read_csv('All_Pitchers_2020_2024_with_Scores_Rounded.csv')

# Clean data: Ensure numeric columns are properly typed, handle missing values
numeric_cols = ['WAR', 'W', 'L', 'ERA', 'G', 'GS', 'SV', 'IP', 'H', 'R', 'ER', 'HR', 'BB', 'SO',
                'WHIP', 'ERA+', 'FIP', 'SO/BB', 'AAV']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Define features for scoring
features = ['WAR', 'ERA+', 'FIP', 'IP', 'ERA', 'WHIP', 'SO', 'SV', 'W', 'SO/BB']
weights = {'WAR': 0.30, 'ERA+': 0.10, 'FIP': 0.10, 'IP': 0.10,
           'ERA': 0.05, 'WHIP': 0.05, 'SO': 0.05, 'SV': 0.05, 'W': 0.05, 'SO/BB': 0.05}

# Normalize features to 0-1 scale (except WAR, which is used directly)
normalized_df = df.copy()
for feature in features:
    if feature != 'WAR':
        if feature in ['ERA', 'WHIP', 'FIP']:  # Invert for metrics where lower is better
            if df[feature].max() > df[feature].min():
                normalized_df[feature] = (df[feature].max() - df[feature]) / (df[feature].max() - df[feature].min())
            else:
                normalized_df[feature] = 0
        else:
            if df[feature].max() > df[feature].min():
                normalized_df[feature] = (df[feature] - df[feature].min()) / (df[feature].max() - df[feature].min())
            else:
                normalized_df[feature] = 0

# Compute cost-effectiveness: WAR per million dollars of AAV
df['CostEffectiveness'] = df['WAR'] / (df['AAV'] / 1_000_000).replace(0, 1)
# Cap extreme values to prevent skewing
df['CostEffectiveness'] = df['CostEffectiveness'].clip(lower=0, upper=df['CostEffectiveness'].quantile(0.95))
# Normalize CostEffectiveness
if df['CostEffectiveness'].max() > df['CostEffectiveness'].min():
    normalized_df['CostEffectiveness'] = (df['CostEffectiveness'] - df['CostEffectiveness'].min()) / \
                                        (df['CostEffectiveness'].max() - df['CostEffectiveness'].min())
else:
    normalized_df['CostEffectiveness'] = 0
weights['CostEffectiveness'] = 0.10

# Define recency weights for years
year_weights = {2024: 0.40, 2023: 0.25, 2022: 0.20, 2021: 0.10, 2020: 0.05}

# Aggregate data by player, applying recency weights
player_scores = []
for player in df['Player'].unique():
    player_data = df[df['Player'] == player].copy()
    if len(player_data) == 1:
        # Single-year player: Use normalized stats directly
        player_normalized = normalized_df[df['Player'] == player]
        composite_score = sum(weights[feat] * player_normalized[feat].iloc[0] for feat in weights)
    else:
        # Multi-year player: Weight stats by year
        weighted_stats = {feat: 0 for feat in weights}
        total_weight = sum(year_weights.get(year, 0) for year in player_data['Year'])
        if total_weight == 0:
            total_weight = 1  # Avoid division by zero
        for _, row in player_data.iterrows():
            year = row['Year']
            weight = year_weights.get(year, 0) / total_weight
            for feat in weights:
                weighted_stats[feat] += weight * normalized_df.loc[row.name, feat]
        composite_score = sum(weights[feat] * weighted_stats[feat] for feat in weights)
    player_scores.append({'Player': player, 'CompositeScore': composite_score})

# Create DataFrame of composite scores
scores_df = pd.DataFrame(player_scores)

# Transform scores to normal distribution (1-99)
mean_score = scores_df['CompositeScore'].mean()
std_score = scores_df['CompositeScore'].std()
scores_df['Z'] = (scores_df['CompositeScore'] - mean_score) / std_score
scores_df['Score_1_100'] = norm.cdf(scores_df['Z']) * 98 + 1  # Scale to 1-99
scores_df['Score_1_100'] = scores_df['Score_1_100'].round().astype(int)
scores_df['Score_1_100'] = scores_df['Score_1_100'].clip(1, 99)  # Ensure within bounds

# Merge scores back to original DataFrame
df = df.drop(columns=['Score_1_100'])
df = df.merge(scores_df[['Player', 'Score_1_100']], on='Player', how='left')

# Export the updated CSV
df.to_csv('All_Pitchers_2020_2024_with_Updated_Scores.csv', index=False)

# For artifact: Return the updated CSV content as a string
with open('All_Pitchers_2020_2024_with_Updated_Scores.csv', 'r') as f:
    updated_csv_content = f.read()

print(updated_csv_content)

Rk,Player,Age,Team,Lg,WAR,W,L,W-L%,ERA,G,GS,GF,CG,SHO,SV,IP,H,R,ER,HR,BB,IBB,SO,HBP,BK,WP,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,AAV,Year,Unnamed: 0,Year_Weight,Weighted_WAR,Salary_per_IP,CostEffectiveness,Score_1_100
1,Albert Abreu,24,NYY,AL,-0.3,0,1,0.0,20.25,2,0,1,0,0,0,1.1,4,4,3,1,2,0,2,1,0,0,11,27.0,16.69,4.5,27.0,6.8,13.5,13.5,1.0,3500.0,2020,,1,-0.3,3181.818181818181,0.0,43
2,Bryan Abreu,23,HOU,AL,0.0,0,0,,2.7,4,0,1,0,0,0,3.1,1,2,1,0,7,0,3,2,0,0,20,181.0,9.49,2.4,2.7,0.0,18.9,8.1,0.43,3500.0,2020,,1,0.0,1129.032258064516,0.0,92
3,Jason Adam,28,CHC,NL,0.1,2,1,0.667,3.29,13,0,5,0,0,0,13.2,9,7,5,2,8,0,21,0,0,0,58,140.0,3.78,1.244,5.9,1.3,5.3,13.8,2.63,109025.0,2020,,1,0.1,8259.469696969698,0.9172208209126348,96
4,Austin Adams,29,SDP,NL,0.0,0,0,,4.5,3,0,1,0,0,0,4.0,3,2,2,1,2,0,7,0,0,1,17,101.0,4.44,1.25,6.8,2.3,4.5,15.8,3.5,91524.0,2020,,1,0.0,22881.0,0.0,45
5,Chance Adams,25,KCR,AL,-0.2,0,0,,9.35,6,0,1,0,0,0,8.2,15,9,9,1,0,0,6,0,0,2,40,52.0,3.31,1.731,15.6,1.0,0.0,6.2,0.0,150096.0,2

In [2]:
pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 60.0 MB/s eta 0:00:00


In [9]:
### Optimizing set of thirteen MLB hitters using MAD and PuLP
import pandas as pd
import numpy as np
import asyncio
import platform
import sys
import os
import logging

# Configuration
CONFIG = {
    'TOTAL_SALARY_CAP': 165000000,  # Realistic MLB roster budget ($100M)
    'YEAR_WEIGHTS': {2020: 0.05, 2021: 0.1, 2022: 0.2, 2023: 0.3, 2024: 0.4},  # Recent years weighted more
    'POSITION_REQUIREMENTS': {
        '2': 1,  # Catcher
        '3': 1,  # First base
        '4': 1,  # Second base
        '5': 1,  # Third base
        '6': 1,  # Shortstop
        '7': 1,  # Left field
        '8': 1,  # Center field
        '9': 1,  # Right field
        'D': 1,  # Designated hitter
        'Utility': 4  # Flexible positions
    }
}

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load data function to read the actual CSV file
def loadFileData(filename):
    logger.info(f"Attempting to load file: {filename}")
    try:
        if os.path.exists(filename):
            with open(filename, 'r', encoding='utf-8') as file:
                logger.info(f"Successfully loaded {filename}")
                return file.read()
        else:
            logger.warning(f"{filename} not found. Using sample dataset instead.")
            return SAMPLE_CSV_DATA
    except Exception as e:
        logger.error(f"Error loading file {filename}: {e}. Using sample dataset.")
        return SAMPLE_CSV_DATA

# Asynchronous main function for Pyodide compatibility
async def main():
    try:
        # Load and parse the CSV data
        csv_data = loadFileData("All_Hitters_2020_2024_with_Updated_Scores.csv")
        df = parse_csv_data(csv_data)

        if df.empty:
            logger.error("No players available after parsing. Exiting.")
            return "Error: No players available after parsing."

        # Initialize agents
        agents = [
            HighWARAgent(),
            ClutchAgent(),
            BangForBuckAgent(),
            TraditionalAgent(),
            PowerAgent()
        ]

        # Each agent selects their top 13 players
        agent_selections = {agent.name: agent.select_players(df) for agent in agents}

        # Debate and optimize to select the final 13 players within total AAV cap
        final_selection, voting_details = debate_and_optimize(df, agent_selections, CONFIG['TOTAL_SALARY_CAP'])

        # Format and print the result
        result = format_result(final_selection, df, CONFIG['TOTAL_SALARY_CAP'], agent_selections, voting_details)
        print(result)

        return result
    except Exception as e:
        logger.error(f"Error in main execution: {e}")
        return f"Error: {e}"

# CSV parsing function
def parse_csv_data(csv_data):
    logger.info("Parsing CSV data")
    try:
        # Parse CSV string using pandas
        from io import StringIO
        df = pd.read_csv(StringIO(csv_data))

        # Clean and preprocess data
        df = df.dropna(subset=['Player', 'WAR', 'G', 'PA', 'BA', 'HR', 'RBI', 'AAV', 'Year'])

        # Convert 'Year' to numeric, coercing errors to NaN
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

        # Replace invalid or missing 'Year' values with a default (e.g., 2024)
        df['Year'] = df['Year'].fillna(2024).astype(int)

        # Ensure 'Year' is within expected range (2020–2024)
        df = df[df['Year'].between(2020, 2024)]

        numeric_cols = ['WAR', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO',
                        'BA', 'OBP', 'SLG', 'OPS', 'OPS+', 'rOBA', 'Rbat+', 'TB', 'GIDP', 'HBP', 'SH', 'SF',
                        'IBB', 'AAV', 'Year', 'Year_Weight', 'Weighted_WAR', 'Salary_per_G', 'Score_1_100']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Apply year-based weighting
        year_weights = CONFIG['YEAR_WEIGHTS']
        df['Weight'] = df['Year'].map(year_weights).fillna(0.05)  # Use 0.05 as default weight
        df['Weight'] = df['Weight'].clip(lower=0.05)  # Ensure weights are at least 0.05
        df = df.dropna(subset=['Weight'])  # Drop rows with NaN weights

        # Log players with low or zero weights for debugging
        low_weight_players = df[df['Weight'] <= 0.05]['Player'].unique()
        if len(low_weight_players) > 0:
            logger.warning(f"Players with low or default weights: {low_weight_players}")

        # Aggregate player stats across years, weighted by year
        def safe_weighted_average(x, weights):
            weights = weights.fillna(0.05).clip(lower=0.05)  # Ensure non-zero weights
            if weights.sum() == 0:
                logger.warning(f"Zero weight sum for player data: {x.name}. Using unweighted average.")
                return x.mean()  # Fallback to unweighted average
            return np.average(x, weights=weights)

        agg_funcs = {
            'WAR': lambda x: safe_weighted_average(x, df.loc[x.index, 'Weight']),
            'Weighted_WAR': lambda x: safe_weighted_average(x, df.loc[x.index, 'Weight']),
            'G': 'sum',
            'PA': 'sum',
            'AB': 'sum',
            'R': 'sum',
            'H': 'sum',
            'HR': 'sum',
            'RBI': 'sum',
            'BA': lambda x: safe_weighted_average(x, df.loc[x.index, 'PA']),
            'SLG': lambda x: safe_weighted_average(x, df.loc[x.index, 'PA']),
            'OPS': lambda x: safe_weighted_average(x, df.loc[x.index, 'PA']),
            'OPS+': lambda x: safe_weighted_average(x, df.loc[x.index, 'PA']),
            'rOBA': lambda x: safe_weighted_average(x, df.loc[x.index, 'PA']),
            'AAV': lambda x: safe_weighted_average(x, df.loc[x.index, 'G']),
            'Score_1_100': lambda x: safe_weighted_average(x, df.loc[x.index, 'PA']),
            'Pos': lambda x: ', '.join(set(x.dropna())),
            'Year': 'count'  # Number of seasons
        }
        df_agg = df.groupby('Player').agg(agg_funcs).reset_index()

        # Filter players with sufficient plate appearances and games played
        df_agg = df_agg[(df_agg['PA'] >= 500) & (df_agg['G'] >= 100)]  # Ensure consistent availability

        logger.info(f"Parsed {len(df_agg)} players after filtering")
        return df_agg
    except Exception as e:
        logger.error(f"Error parsing CSV data: {e}")
        return pd.DataFrame()

# Base Agent class
class Agent:
    def __init__(self, name):
        self.name = name

    def select_players(self, df):
        raise NotImplementedError

# High WAR Agent
class HighWARAgent(Agent):
    def __init__(self):
        super().__init__("High WAR Agent")

    def select_players(self, df):
        sorted_df = df.sort_values(by='Weighted_WAR', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Weighted_WAR']]
        max_war = sorted_df['Weighted_WAR'].max()
        top_players['Confidence'] = top_players['Weighted_WAR'] / max_war
        logger.info(f"{self.name} selected {len(top_players)} players")
        return top_players

# Clutch Agent
class ClutchAgent(Agent):
    def __init__(self):
        super().__init__("Clutch Agent")

    def select_players(self, df):
        df['Clutch_Score'] = (df['OPS+'] * 0.5 + df['Score_1_100'] * 0.5)
        sorted_df = df.sort_values(by='Clutch_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Clutch_Score']]
        max_score = sorted_df['Clutch_Score'].max()
        top_players['Confidence'] = top_players['Clutch_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} players")
        return top_players

# Bang for Buck Agent
class BangForBuckAgent(Agent):
    def __init__(self):
        super().__init__("Bang for Buck Agent")

    def select_players(self, df):
        df['Value_Score'] = (df['Weighted_WAR'] * 0.5 + df['OPS'] * 100 * 0.3 + df['HR'] * 0.2) / (df['AAV'] + 1) * 1e6
        sorted_df = df.sort_values(by='Value_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Value_Score']]
        max_score = sorted_df['Value_Score'].max()
        top_players['Confidence'] = top_players['Value_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} players")
        return top_players

# Traditional Agent
class TraditionalAgent(Agent):
    def __init__(self):
        super().__init__("Traditional Agent")

    def select_players(self, df):
        df['Traditional_Score'] = (
            df['BA'] / df['BA'].max() * 0.4 +
            df['HR'] / df['HR'].max() * 0.3 +
            df['RBI'] / df['RBI'].max() * 0.3
        )
        sorted_df = df.sort_values(by='Traditional_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Traditional_Score']]
        max_score = sorted_df['Traditional_Score'].max()
        top_players['Confidence'] = top_players['Traditional_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} players")
        return top_players

# Power Agent
class PowerAgent(Agent):
    def __init__(self):
        super().__init__("Power Agent")

    def select_players(self, df):
        df['Power_Score'] = (
            df['HR'] / df['HR'].max() * 0.5 +
            df['SLG'] / df['SLG'].max() * 0.5
        )
        sorted_df = df.sort_values(by='Power_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Power_Score']]
        max_score = sorted_df['Power_Score'].max()
        top_players['Confidence'] = top_players['Power_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} players")
        return top_players

# Debate and optimization function with total AAV cap
def debate_and_optimize(df, agent_selections, total_salary_cap):
    logger.info("Starting optimization")
    try:
        # Combine all selections
        all_players = set()
        for selections in agent_selections.values():
            all_players.update(selections['Player'])

        # Create a score matrix
        player_scores = {}
        for player in all_players:
            total_score = 0
            for agent_name, selections in agent_selections.items():
                player_row = selections[selections['Player'] == player]
                if not player_row.empty:
                    confidence = player_row['Confidence'].iloc[0]
                    total_score += confidence
                else:
                    total_score += -0.1
            player_scores[player] = total_score

        # Track voting details for all players in the dataset
        voting_details = {player: {} for player in df['Player']}
        for agent_name, selections in agent_selections.items():
            for _, row in selections.iterrows():
                player = row['Player']
                confidence = row['Confidence']
                voting_details[player][agent_name] = confidence

        if PULP_AVAILABLE:
            # Linear programming optimization
            prob = LpProblem("Roster_Optimization", LpMaximize)

            # Variables: 1 if player is selected, 0 otherwise
            player_vars = {player: LpVariable(f"Player_{player}", cat='Binary') for player in df['Player']}

            # Objective: Maximize total score
            prob += lpSum(player_scores.get(player, 0) * player_vars[player] for player in df['Player'])

            # Constraint: Exactly 13 players
            prob += lpSum(player_vars[player] for player in df['Player']) == 13, "Total_Players"

            # Constraint: Salary cap
            prob += lpSum(df[df['Player'] == player]['AAV'].iloc[0] * player_vars[player] for player in df['Player']) <= total_salary_cap, "Salary_Cap"

            # Positional constraints
            for pos, count in CONFIG['POSITION_REQUIREMENTS'].items():
                if pos != 'Utility':
                    prob += lpSum(player_vars[player] for player in df[df['Pos'].str.contains(pos, na=False)]['Player']) >= count, f"Min_{pos}"
                else:
                    # Utility players can fill any position
                    prob += lpSum(player_vars[player] for player in df['Player']) >= count, "Utility"

            # Solve the problem
            prob.solve()

            if LpStatus[prob.status] == 'Optimal':
                selected_players = [player for player in df['Player'] if player_vars[player].varValue > 0.5]
                selected_voting_details = {player: voting_details[player] for player in selected_players}
                logger.info(f"Selected {len(selected_players)} players using PuLP")
                return selected_players[:13], selected_voting_details

        logger.warning("No optimal solution found or PuLP unavailable. Falling back to greedy selection.")
        return greedy_fallback(df, agent_selections, total_salary_cap, voting_details)
    except Exception as e:
        logger.error(f"Error in optimization: {e}. Falling back to greedy selection.")
        return greedy_fallback(df, agent_selections, total_salary_cap, voting_details)

# Greedy fallback if LP fails or PuLP is unavailable
def greedy_fallback(df, agent_selections, total_salary_cap, voting_details):
    logger.info("Running greedy fallback selection")
    all_players = set()
    for selections in agent_selections.values():
        all_players.update(selections['Player'])

    player_scores = {}
    for player in all_players:
        total_score = 0
        for agent_name, selections in agent_selections.items():
            player_row = selections[selections['Player'] == player]
            if not player_row.empty:
                confidence = player_row['Confidence'].iloc[0]
                total_score += confidence
            else:
                total_score += -0.1
        player_scores[player] = total_score

    sorted_players = sorted(player_scores.items(), key=lambda x: x[1], reverse=True)
    candidate_players = [p[0] for p in sorted_players]

    selected_players = []
    total_aav = 0
    required_positions = CONFIG['POSITION_REQUIREMENTS']
    covered_positions = {pos: 0 for pos in required_positions}

    candidate_df = df[df['Player'].isin(candidate_players)].copy()
    candidate_df['Score'] = candidate_df['Player'].map(player_scores)
    candidate_df['Score_per_AAV'] = candidate_df['Score'] / (candidate_df['AAV'] + 1)
    candidate_df = candidate_df.sort_values(by='Score_per_AAV', ascending=False)

    for _, row in candidate_df.iterrows():
        player = row['Player']
        aav = row['AAV']
        if len(selected_players) < 13 and total_aav + aav <= total_salary_cap:
            player_positions = row['Pos'].split(', ') if pd.notna(row['Pos']) else []
            can_add = False
            for pos in player_positions:
                if pos in required_positions and covered_positions[pos] < required_positions[pos]:
                    can_add = True
                    covered_positions[pos] += 1
                    break
            if can_add or covered_positions['Utility'] < required_positions['Utility']:
                selected_players.append(player)
                total_aav += aav
                if not can_add:
                    covered_positions['Utility'] += 1

    if len(selected_players) < 13:
        remaining_df = df[~df['Player'].isin(selected_players)].copy()
        remaining_df['Score'] = remaining_df['Player'].map(player_scores)
        remaining_df = remaining_df[remaining_df['AAV'] + total_aav <= total_salary_cap]
        remaining_df = remaining_df.sort_values(by='Score', ascending=False)

        for _, row in remaining_df.iterrows():
            if len(selected_players) < 13:
                player = row['Player']
                aav = row['AAV']
                if total_aav + aav <= total_salary_cap:
                    selected_players.append(player)
                    total_aav += aav
                    covered_positions['Utility'] += 1

    selected_voting_details = {player: voting_details[player] for player in selected_players}
    logger.info(f"Selected {len(selected_players)} players using greedy fallback")
    return selected_players[:13], selected_voting_details

# Format the result with voting details
def format_result(final_selection, df, total_salary_cap, agent_selections, voting_details):
    result = f"Final Selection of Top 13 Hitters (Total AAV Cap: ${total_salary_cap:,.2f}):\n\n"
    selected_df = df[df['Player'].isin(final_selection)]
    total_aav = selected_df['AAV'].sum()

    for _, row in selected_df.iterrows():
        player = row['Player']
        result += (
            f"Player: {row['Player']}\n"
            f"WAR: {row['Weighted_WAR']:.2f}\n"
            f"BA: {row['BA']:.3f}\n"
            f"HR: {row['HR']:.0f}\n"
            f"RBI: {row['RBI']:.0f}\n"
            f"OPS+: {row['OPS+']:.0f}\n"
            f"AAV: ${row['AAV']:.2f}\n"
            f"Positions: {row['Pos']}\n"
            f"Votes:\n"
        )
        player_votes = voting_details.get(player, {})
        for agent_name in agent_selections.keys():
            if agent_name in player_votes:
                result += f"  - {agent_name}: Confidence = {player_votes[agent_name]:.2f}\n"
            else:
                result += f"  - {agent_name}: Not selected\n"
        result += (
            f"Rationale: Selected for balanced performance across WAR, traditional stats, power, clutch ability, and cost-effectiveness within AAV cap.\n\n"
        )

    result += (
        f"Total AAV: ${total_aav:,.2f}\n"
        f"Salary Cap Compliance: {'Met' if total_aav <= total_salary_cap else 'Exceeded'}\n"
    )
    return result

# Execution block handling Pyodide, Jupyter/IPython, and standard Python
if platform.system() == "Emscripten":
    # Pyodide environment
    asyncio.ensure_future(main())
else:
    # Check if running in IPython/Jupyter
    if 'IPython' in sys.modules:
        # Jupyter/IPython: Use await in the existing event loop
        import nest_asyncio
        nest_asyncio.apply()  # Allow nested event loops in Jupyter
        loop = asyncio.get_event_loop()
        loop.run_until_complete(main())
    else:
        # Standard Python: Run with asyncio.run
        if __name__ == "__main__":
            asyncio.run(main())

 'Abraham Toro' 'Adalberto Mondesí' 'Adam Duvall' 'Adam Eaton'
 'Adam Engel' 'Adam Frazier' 'Adam Haseley' 'Adam Kolarek'
 'Adeiny Hechavarría' 'Adolis García' 'AJ Pollock' 'Albert Almora'
 'Albert Pujols' 'Alec Bohm' 'Alec Mills' 'Aledmys Díaz' 'Alejandro Kirk'
 'Alex Avila' 'Alex Bregman' 'Alex Dickerson' 'Alex Gordon' 'Alex Jackson'
 'Alex Verdugo' 'Alí Sánchez' 'Amed Rosario' 'Anderson Tejeda'
 'Andrelton Simmons' 'Andrés Giménez' 'Andrew Benintendi' 'Andrew Knapp'
 'Andrew Knizner' 'Andrew McCutchen' 'Andrew Romine' 'Andrew Stevenson'
 'Andrew Susac' 'Andrew Velazquez' 'Andrew Young' 'Anthony Alford'
 'Anthony Bemboom' 'Anthony Rendon' 'Anthony Rizzo' 'Anthony Santander'
 'Aristides Aquino' 'Asdrúbal Cabrera' 'Austin Adams' 'Austin Allen'
 'Austin Barnes' 'Austin Davis' 'Austin Dean' 'Austin Hays'
 'Austin Hedges' 'Austin Meadows' 'Austin Nola' 'Austin Riley'
 'Austin Romine' 'Austin Slater' 'Avisaíl García' 'Beau Taylor'
 'Ben Gamel' 'Billy Hamilton' 'Billy McKinney' 'Bo Bichette

Final Selection of Top 13 Hitters (Total AAV Cap: $165,000,000.00):

Player: Aaron Judge
WAR: 33.18
BA: 0.298
HR: 205
RBI: 470
OPS+: 190
AAV: $25355189.52
Positions: 9/DH
Votes:
  - High WAR Agent: Confidence = 1.00
  - Clutch Agent: Confidence = 1.00
  - Bang for Buck Agent: Not selected
  - Traditional Agent: Confidence = 0.97
  - Power Agent: Confidence = 1.00
Rationale: Selected for balanced performance across WAR, traditional stats, power, clutch ability, and cost-effectiveness within AAV cap.

Player: Freddie Freeman
WAR: 21.34
BA: 0.313
HR: 116
RBI: 427
OPS+: 153
AAV: $19456033.35
Positions: *3/DH
Votes:
  - High WAR Agent: Confidence = 0.64
  - Clutch Agent: Confidence = 0.87
  - Bang for Buck Agent: Not selected
  - Traditional Agent: Confidence = 0.83
  - Power Agent: Not selected
Rationale: Selected for balanced performance across WAR, traditional stats, power, clutch ability, and cost-effectiveness within AAV cap.

Player: Jazz Chisholm Jr.
WAR: 8.07
BA: 0.249
HR: 77
RBI: 2

In [12]:
### Optimizing set of thirteen MLB pitchers using MAD and PuLP
import pandas as pd
import numpy as np
import asyncio
import platform
import sys
import os
import logging

# Configuration
CONFIG = {
    'TOTAL_SALARY_CAP': 165000000,  # Realistic MLB pitching staff budget ($100M)
    'YEAR_WEIGHTS': {2020: 0.05, 2021: 0.1, 2022: 0.2, 2023: 0.3, 2024: 0.4},  # Recent years weighted more
    'ROLE_REQUIREMENTS': {
        'Starter': 5,  # Pitchers with GS ≥ 10
        'Reliever': 8  # Pitchers with GS < 10
    }
}

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load data function to read the actual CSV file
def loadFileData(filename):
    logger.info(f"Attempting to load file: {filename}")
    try:
        if os.path.exists(filename):
            with open(filename, 'r', encoding='utf-8') as file:
                logger.info(f"Successfully loaded {filename}")
                return file.read()
        else:
            logger.warning(f"{filename} not found. Using sample dataset instead.")
            return SAMPLE_CSV_DATA
    except Exception as e:
        logger.error(f"Error loading file {filename}: {e}. Using sample dataset.")
        return SAMPLE_CSV_DATA

# Asynchronous main function for Pyodide compatibility
async def main():
    try:
        # Load and parse the CSV data
        csv_data = loadFileData("All_Pitchers_2020_2024_with_Updated_Scores.csv")
        df = parse_csv_data(csv_data)

        if df.empty:
            logger.error("No pitchers available after parsing. Exiting.")
            return "Error: No pitchers available after parsing."

        # Initialize agents
        agents = [
            FIPAgent(),
            WARAgent(),
            StrikeoutAgent(),
            TraditionalAgent(),
            BangForBuckAgent()
        ]

        # Each agent selects their top 13 pitchers
        agent_selections = {agent.name: agent.select_players(df) for agent in agents}

        # Debate and optimize to select the final 13 pitchers within total AAV cap
        final_selection, voting_details = debate_and_optimize(df, agent_selections, CONFIG['TOTAL_SALARY_CAP'])

        # Format and print the result
        result = format_result(final_selection, df, CONFIG['TOTAL_SALARY_CAP'], agent_selections, voting_details)
        print(result)

        return result
    except Exception as e:
        logger.error(f"Error in main execution: {e}")
        return f"Error: {e}"

# CSV parsing function
def parse_csv_data(csv_data):
    logger.info("Parsing CSV data")
    try:
        # Parse CSV string using pandas
        from io import StringIO
        df = pd.read_csv(StringIO(csv_data))

        # Log initial DataFrame shape and columns
        logger.info(f"Initial DataFrame shape: {df.shape}")
        logger.info(f"Columns in CSV: {list(df.columns)}")

        # Clean and preprocess data
        required_cols = ['Player', 'WAR', 'G', 'IP', 'ERA', 'FIP', 'SO9', 'SO/BB', 'AAV', 'Year']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            logger.error(f"Missing required columns: {missing_cols}")
            return pd.DataFrame()

        df = df.dropna(subset=required_cols)
        logger.info(f"DataFrame shape after dropna: {df.shape}")

        numeric_cols = ['WAR', 'W', 'L', 'ERA', 'G', 'GS', 'GF', 'CG', 'SHO', 'SV', 'IP', 'H', 'R', 'ER',
                        'HR', 'BB', 'IBB', 'SO', 'HBP', 'BK', 'WP', 'BF', 'ERA+', 'FIP', 'WHIP', 'H9', 'HR9',
                        'BB9', 'SO9', 'SO/BB', 'AAV', 'Year', 'Year_Weight', 'Weighted_WAR', 'Salary_per_IP',
                        'Score_1_100']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            else:
                logger.warning(f"Column {col} not found in DataFrame")

        # Apply year-based weighting
        year_weights = CONFIG['YEAR_WEIGHTS']
        df['Weight'] = df['Year'].map(year_weights).fillna(0.1)  # Default weight for unexpected years

        # Aggregate pitcher stats across years, weighted by year
        agg_funcs = {
            'WAR': lambda x: np.average(x, weights=df.loc[x.index, 'Weight']),
            'Weighted_WAR': lambda x: np.average(x, weights=df.loc[x.index, 'Weight']),
            'G': 'sum',
            'GS': 'sum',
            'IP': 'sum',
            'W': 'sum',
            'L': 'sum',
            'ERA': lambda x: np.average(x, weights=df.loc[x.index, 'IP']),
            'FIP': lambda x: np.average(x, weights=df.loc[x.index, 'IP']),
            'WHIP': lambda x: np.average(x, weights=df.loc[x.index, 'IP']),
            'SO9': lambda x: np.average(x, weights=df.loc[x.index, 'IP']),
            'SO/BB': lambda x: np.average(x, weights=df.loc[x.index, 'IP']),
            'AAV': lambda x: np.average(x, weights=df.loc[x.index, 'G']),
            'Score_1_100': lambda x: np.average(x, weights=df.loc[x.index, 'IP']),
            'Year': 'count'  # Number of seasons
        }
        df_agg = df.groupby('Player').agg(agg_funcs).reset_index()

        # Filter pitchers with sufficient innings pitched and games
        df_agg = df_agg[(df_agg['IP'] >= 50) & (df_agg['G'] >= 10)]  # Ensure consistent availability
        logger.info(f"Parsed {len(df_agg)} pitchers after filtering")

        if df_agg.empty:
            logger.warning("No pitchers remain after filtering. Check IP and G thresholds.")

        return df_agg
    except Exception as e:
        logger.error(f"Error parsing CSV data: {e}")
        return pd.DataFrame()

# Base Agent class
class Agent:
    def __init__(self, name):
        self.name = name

    def select_players(self, df):
        raise NotImplementedError

# FIP Agent
class FIPAgent(Agent):
    def __init__(self):
        super().__init__("FIP Agent")

    def select_players(self, df):
        sorted_df = df.sort_values(by='FIP', ascending=True)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'FIP']]
        min_fip = sorted_df['FIP'].min()
        top_players['Confidence'] = (sorted_df['FIP'].max() - top_players['FIP']) / (sorted_df['FIP'].max() - min_fip)
        logger.info(f"{self.name} selected {len(top_players)} pitchers")
        return top_players

# WAR Agent
class WARAgent(Agent):
    def __init__(self):
        super().__init__("WAR Agent")

    def select_players(self, df):
        sorted_df = df.sort_values(by='Weighted_WAR', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Weighted_WAR']]
        max_war = sorted_df['Weighted_WAR'].max()
        top_players['Confidence'] = top_players['Weighted_WAR'] / max_war
        logger.info(f"{self.name} selected {len(top_players)} pitchers")
        return top_players

# Strikeout Agent
class StrikeoutAgent(Agent):
    def __init__(self):
        super().__init__("Strikeout Agent")

    def select_players(self, df):
        df['Strikeout_Score'] = (df['SO9'] * 0.6 + df['SO/BB'] * 0.4)
        sorted_df = df.sort_values(by='Strikeout_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Strikeout_Score']]
        max_score = sorted_df['Strikeout_Score'].max()
        top_players['Confidence'] = top_players['Strikeout_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} pitchers")
        return top_players

# Traditional Agent
class TraditionalAgent(Agent):
    def __init__(self):
        super().__init__("Traditional Agent")

    def select_players(self, df):
        df['Traditional_Score'] = (
            (df['W'] / df['W'].max() * 0.3) +
            ((df['L'].max() - df['L']) / df['L'].max() * 0.3) +
            ((df['ERA'].max() - df['ERA']) / df['ERA'].max() * 0.2) +
            ((df['WHIP'].max() - df['WHIP']) / df['WHIP'].max() * 0.2)
        )
        sorted_df = df.sort_values(by='Traditional_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Traditional_Score']]
        max_score = sorted_df['Traditional_Score'].max()
        top_players['Confidence'] = top_players['Traditional_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} pitchers")
        return top_players

# Bang for Buck Agent
class BangForBuckAgent(Agent):
    def __init__(self):
        super().__init__("Bang for Buck Agent")

    def select_players(self, df):
        df['Value_Score'] = (
            (df['Weighted_WAR'] * 0.4 + df['SO9'] * 0.3 + (df['ERA'].max() - df['ERA']) / df['ERA'].max() * 0.3)
            / (df['AAV'] + 1) * 1e6
        )
        sorted_df = df.sort_values(by='Value_Score', ascending=False)
        top_players = sorted_df.head(min(13, len(sorted_df)))[['Player', 'Value_Score']]
        max_score = sorted_df['Value_Score'].max()
        top_players['Confidence'] = top_players['Value_Score'] / max_score
        logger.info(f"{self.name} selected {len(top_players)} pitchers")
        return top_players

# Debate and optimization function with total AAV cap
def debate_and_optimize(df, agent_selections, total_salary_cap):
    logger.info("Starting optimization")
    try:
        # Combine all selections
        all_players = set()
        for selections in agent_selections.values():
            all_players.update(selections['Player'])

        # Create a score matrix
        player_scores = {}
        for player in all_players:
            total_score = 0
            for agent_name, selections in agent_selections.items():
                player_row = selections[selections['Player'] == player]
                if not player_row.empty:
                    confidence = player_row['Confidence'].iloc[0]
                    total_score += confidence
                else:
                    total_score += -0.1
            player_scores[player] = total_score

        # Track voting details for all players in the dataset
        voting_details = {player: {} for player in df['Player']}
        for agent_name, selections in agent_selections.items():
            for _, row in selections.iterrows():
                player = row['Player']
                confidence = row['Confidence']
                voting_details[player][agent_name] = confidence

        if PULP_AVAILABLE:
            # Linear programming optimization
            prob = LpProblem("Pitching_Staff_Optimization", LpMaximize)

            # Variables: 1 if pitcher is selected, 0 otherwise
            player_vars = {player: LpVariable(f"Player_{player}", cat='Binary') for player in df['Player']}

            # Objective: Maximize total score
            prob += lpSum(player_scores.get(player, 0) * player_vars[player] for player in df['Player'])

            # Constraint: Exactly 13 pitchers
            prob += lpSum(player_vars[player] for player in df['Player']) == 13, "Total_Pitchers"

            # Constraint: Salary cap
            prob += lpSum(df[df['Player'] == player]['AAV'].iloc[0] * player_vars[player] for player in df['Player']) <= total_salary_cap, "Salary_Cap"

            # Role constraints
            prob += lpSum(player_vars[player] for player in df[df['GS'] >= 10]['Player']) == CONFIG['ROLE_REQUIREMENTS']['Starter'], "Starters"
            prob += lpSum(player_vars[player] for player in df[df['GS'] < 10]['Player']) == CONFIG['ROLE_REQUIREMENTS']['Reliever'], "Relievers"

            # Solve the problem
            prob.solve()

            if LpStatus[prob.status] == 'Optimal':
                selected_players = [player for player in df['Player'] if player_vars[player].varValue > 0.5]
                selected_voting_details = {player: voting_details[player] for player in selected_players}
                logger.info(f"Selected {len(selected_players)} pitchers using PuLP")
                return selected_players[:13], selected_voting_details

        logger.warning("No optimal solution found or PuLP unavailable. Falling back to greedy selection.")
        return greedy_fallback(df, agent_selections, total_salary_cap, voting_details)
    except Exception as e:
        logger.error(f"Error in optimization: {e}. Falling back to greedy selection.")
        return greedy_fallback(df, agent_selections, total_salary_cap, voting_details)

# Greedy fallback if LP fails or PuLP is unavailable
def greedy_fallback(df, agent_selections, total_salary_cap, voting_details):
    logger.info("Running greedy fallback selection")
    all_players = set()
    for selections in agent_selections.values():
        all_players.update(selections['Player'])

    player_scores = {}
    for player in all_players:
        total_score = 0
        for agent_name, selections in agent_selections.items():
            player_row = selections[selections['Player'] == player]
            if not player_row.empty:
                confidence = player_row['Confidence'].iloc[0]
                total_score += confidence
            else:
                total_score += -0.1
        player_scores[player] = total_score

    sorted_players = sorted(player_scores.items(), key=lambda x: x[1], reverse=True)
    candidate_players = [p[0] for p in sorted_players]

    selected_players = []
    total_aav = 0
    required_roles = CONFIG['ROLE_REQUIREMENTS']
    covered_roles = {'Starter': 0, 'Reliever': 0}

    candidate_df = df[df['Player'].isin(candidate_players)].copy()
    candidate_df['Score'] = candidate_df['Player'].map(player_scores)
    candidate_df['Score_per_AAV'] = candidate_df['Score'] / (candidate_df['AAV'] + 1)
    candidate_df = candidate_df.sort_values(by='Score_per_AAV', ascending=False)

    for _, row in candidate_df.iterrows():
        player = row['Player']
        aav = row['AAV']
        is_starter = row['GS'] >= 10
        role = 'Starter' if is_starter else 'Reliever'
        if len(selected_players) < 13 and total_aav + aav <= total_salary_cap:
            if covered_roles[role] < required_roles[role]:
                selected_players.append(player)
                total_aav += aav
                covered_roles[role] += 1

    # Fill remaining slots if needed
    if len(selected_players) < 13:
        remaining_df = df[~df['Player'].isin(selected_players)].copy()
        remaining_df['Score'] = remaining_df['Player'].map(player_scores)
        remaining_df = remaining_df[remaining_df['AAV'] + total_aav <= total_salary_cap]
        remaining_df = remaining_df.sort_values(by='Score', ascending=False)

        for _, row in remaining_df.iterrows():
            if len(selected_players) < 13:
                player = row['Player']
                aav = row['AAV']
                is_starter = row['GS'] >= 10
                role = 'Starter' if is_starter else 'Reliever'
                if covered_roles[role] < required_roles[role] or sum(covered_roles.values()) < 13:
                    selected_players.append(player)
                    total_aav += aav
                    covered_roles[role] += 1

    selected_voting_details = {player: voting_details[player] for player in selected_players}
    logger.info(f"Selected {len(selected_players)} pitchers using greedy fallback")
    return selected_players[:13], selected_voting_details

# Format the result with voting details
def format_result(final_selection, df, total_salary_cap, agent_selections, voting_details):
    result = f"Final Selection of Top 13 Pitchers (Total AAV Cap: ${total_salary_cap:,.2f}):\n\n"
    selected_df = df[df['Player'].isin(final_selection)]
    total_aav = selected_df['AAV'].sum()

    for _, row in selected_df.iterrows():
        player = row['Player']
        role = 'Starter' if row['GS'] >= 10 else 'Reliever'
        result += (
            f"Player: {row['Player']}\n"
            f"Role: {role}\n"
            f"WAR: {row['Weighted_WAR']:.2f}\n"
            f"ERA: {row['ERA']:.2f}\n"
            f"FIP: {row['FIP']:.2f}\n"
            f"SO/9: {row['SO9']:.2f}\n"
            f"SO/BB: {row['SO/BB']:.2f}\n"
            f"W-L: {row['W']:.0f}-{row['L']:.0f}\n"
            f"WHIP: {row['WHIP']:.3f}\n"
            f"AAV: ${row['AAV']:.2f}\n"
            f"Votes:\n"
        )
        player_votes = voting_details.get(player, {})
        for agent_name in agent_selections.keys():
            if agent_name in player_votes:
                result += f"  - {agent_name}: Confidence = {player_votes[agent_name]:.2f}\n"
            else:
                result += f"  - {agent_name}: Not selected\n"
        result += (
            f"Rationale: Selected for balanced performance across FIP, WAR, strikeouts, traditional stats, and cost-effectiveness within AAV cap.\n\n"
        )

    result += (
        f"Total AAV: ${total_aav:,.2f}\n"
        f"Salary Cap Compliance: {'Met' if total_aav <= total_salary_cap else 'Exceeded'}\n"
    )
    return result

# Execution block handling Pyodide, Jupyter/IPython, and standard Python
if platform.system() == "Emscripten":
    # Pyodide environment
    asyncio.ensure_future(main())
else:
    # Check if running in IPython/Jupyter
    if 'IPython' in sys.modules:
        # Jupyter/IPython: Use await in the existing event loop
        import nest_asyncio
        nest_asyncio.apply()  # Allow nested event loops in Jupyter
        loop = asyncio.get_event_loop()
        loop.run_until_complete(main())
    else:
        # Standard Python: Run with asyncio.run
        if __name__ == "__main__":
            asyncio.run(main())

Final Selection of Top 13 Pitchers (Total AAV Cap: $165,000,000.00):

Player: Cade Smith
Role: Reliever
WAR: 12.00
ERA: 1.91
FIP: 1.40
SO/9: 12.30
SO/BB: 6.06
W-L: 6-1
WHIP: 0.903
AAV: $1297688.00
Votes:
  - FIP Agent: Confidence = 1.00
  - WAR Agent: Not selected
  - Strikeout Agent: Confidence = 0.77
  - Traditional Agent: Not selected
  - Bang for Buck Agent: Not selected
Rationale: Selected for balanced performance across FIP, WAR, strikeouts, traditional stats, and cost-effectiveness within AAV cap.

Player: Devin Williams
Role: Reliever
WAR: 7.30
ERA: 1.70
FIP: 2.24
SO/9: 14.62
SO/BB: 3.51
W-L: 27-10
WHIP: 0.977
AAV: $2056026.68
Votes:
  - FIP Agent: Confidence = 0.86
  - WAR Agent: Not selected
  - Strikeout Agent: Confidence = 0.80
  - Traditional Agent: Confidence = 0.98
  - Bang for Buck Agent: Not selected
Rationale: Selected for balanced performance across FIP, WAR, strikeouts, traditional stats, and cost-effectiveness within AAV cap.

Player: Edwin Díaz
Role: Reliever
WAR: